<a href="https://colab.research.google.com/github/gabrielavilladiego2008-sys/-GVG/blob/main/AgroDatos_MySQL_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Instalar dependencias para conectar MySQL en Colab
!pip install -q ipython-sql pymysql
%load_ext sql
%sql mysql+pymysql://root:@localhost/

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/pymysql/connections.py", line 681, in connect
    sock = socket.create_connection(
        (self.host, self.port), self.connect_timeout, **kwargs
    )
  File "/usr/lib/python3.13/socket.py", line 868, in create_connection
    raise exceptions[0]
  File "/usr/lib/python3.13/socket.py", line 853, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
ConnectionRefusedError: [Errno 111] Connection refused

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/sqlalchemy/engine/base.py", line 154, in __init__
    self._dbapi_connection = engine.raw_connection()
                             ~~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/sqlalchemy/engine/base.py", line 3303, in raw_connection
    return self.pool.connect()
           ~~~~~~~~~~~~~~~~~^^
  File "/usr/local/lib/pyt

In [ ]:
%%sql
-- 1. Agregar columna tipo_riego a la tabla lote
ALTER TABLE lote
ADD COLUMN tipo_riego ENUM('goteo', 'aspersión', 'gravedad', 'secano') NOT NULL DEFAULT 'secano';

-- 2. Crear tabla operario
CREATE TABLE operario (
    id_operario INT AUTO_INCREMENT PRIMARY KEY,
    documento VARCHAR(20) NOT NULL UNIQUE,
    nombre_completo VARCHAR(100) NOT NULL,
    telefono VARCHAR(20),
    certificado_aplicador BOOLEAN NOT NULL DEFAULT FALSE
);

-- 3. Insertar tres operarios y mostrar resultado
INSERT INTO operario (documento, nombre_completo, telefono, certificado_aplicador) VALUES
('1012345678', 'Carlos Mendoza', '3101234567', TRUE),
('1098765432', 'Ana Gómez', '3209876543', FALSE),
('1055544433', 'Luis Rodríguez', '3155554444', TRUE);

SELECT * FROM operario;

In [ ]:
%%sql
INSERT INTO producto (
    nombre_comercial, tipo, ingrediente_activo, categoria_toxicologica,
    registro_ica, dosis_ha, unidad_dosis, carencia_dias, reingreso_horas, precio_cop
) VALUES
('BioLarva WP', 'bioinsumo', 'Bacillus thuringiensis var. kurstaki', 'IV', 'ICA-DEMO-2001', 0.50, 'kg', 0, 4, 95000.00),
('RoyaStop 250 EC', 'fungicida', 'Triazol 250 g/L', 'III', 'ICA-DEMO-3001', 0.40, 'L', 30, 24, 160000.00),
('TizónFin 80 WP', 'fungicida', 'Ditiocarbamato 800 g/kg', 'III', 'ICA-DEMO-3002', 2.50, 'kg', 14, 24, 38000.00);

SELECT * FROM producto;

In [ ]:
%%sql
-- 1. Aplicación calculada con subconsulta
INSERT INTO aplicacion (id_lote, id_producto, fecha, cantidad_aplicada, responsable)
SELECT
    l.id_lote,
    p.id_producto,
    CURDATE(),
    (p.dosis_ha * l.area_ha),
    'Carlos Mendoza'
FROM lote l, producto p
WHERE l.codigo = 'POR-L01' AND p.nombre_comercial = 'RoyaStop 250 EC';

-- 2. Fecha mínima de cosecha permitida
SELECT
    a.id_aplicacion,
    l.codigo AS lote,
    p.nombre_comercial AS producto,
    a.fecha AS fecha_aplicacion,
    p.carencia_dias,
    DATE_ADD(a.fecha, INTERVAL p.carencia_dias DAY) AS fecha_minima_cosecha
FROM aplicacion a
JOIN lote l ON a.id_lote = l.id_lote
JOIN producto p ON a.id_producto = p.id_producto;

In [ ]:
%%sql
-- 1. Área total sembrada por cultivo (ordenada de mayor a menor)
SELECT cultivo, SUM(area_ha) AS total_area_ha
FROM lote
GROUP BY cultivo
ORDER BY total_area_ha DESC;

-- 2. Fincas cuya área total sembrada supera 5 ha
SELECT f.nombre AS finca, SUM(l.area_ha) AS total_area
FROM finca f
JOIN lote l ON f.id_finca = l.id_finca
GROUP BY f.id_finca, f.nombre
HAVING total_area > 5;

-- 3. Productos que nunca se han aplicado
SELECT p.id_producto, p.nombre_comercial, p.tipo
FROM producto p
LEFT JOIN aplicacion a ON p.id_producto = a.id_producto
WHERE a.id_aplicacion IS NULL;

-- 4. Responsable que más kilogramos de fertilizante ha aplicado
SELECT a.responsable, SUM(a.cantidad_aplicada) AS total_kg_aplicados
FROM aplicacion a
JOIN producto p ON a.id_producto = p.id_producto
WHERE p.tipo = 'fertilizante' AND p.unidad_dosis = 'kg'
GROUP BY a.responsable
ORDER BY total_kg_aplicados DESC
LIMIT 1;

In [ ]:
%%sql
-- 1. Crear tabla operario_n en 3FN
CREATE TABLE operario_n (
    documento VARCHAR(20) PRIMARY KEY,
    nombre VARCHAR(100) NOT NULL,
    telefono VARCHAR(20)
);

-- 2. Crear tabla aplicacion_n en 3FN
CREATE TABLE aplicacion_n (
    id_aplicacion INT AUTO_INCREMENT PRIMARY KEY,
    fecha DATE NOT NULL,
    id_lote INT NOT NULL,
    id_producto INT NOT NULL,
    cantidad_aplicada DECIMAL(8,2) NOT NULL,
    documento_operario VARCHAR(20) NOT NULL,
    observaciones TEXT,
    FOREIGN KEY (id_lote) REFERENCES lote(id_lote),
    FOREIGN KEY (id_producto) REFERENCES producto(id_producto),
    FOREIGN KEY (documento_operario) REFERENCES operario_n(documento)
);

-- 3. Migrar datos de planilla_campo
INSERT INTO operario_n (documento, nombre, telefono)
SELECT DISTINCT documento_operario, operario, telefono_operario
FROM planilla_campo;

INSERT INTO aplicacion_n (fecha, id_lote, id_producto, cantidad_aplicada, documento_operario, observaciones)
SELECT
    pc.fecha,
    l.id_lote,
    p.id_producto,
    pc.cantidad,
    pc.documento_operario,
    pc.observaciones
FROM planilla_campo pc
JOIN lote l ON pc.codigo_lote = l.codigo
JOIN producto p ON pc.nombre_producto = p.nombre_comercial;

-- 4. Reconstrucción de la planilla original
SELECT
    a.fecha,
    l.codigo AS codigo_lote,
    p.nombre_comercial AS nombre_producto,
    p.categoria_toxicologica AS categoria_tox,
    a.cantidad_aplicada AS cantidad,
    op.nombre AS operario,
    op.documento AS documento_operario,
    op.telefono AS telefono_operario,
    a.observaciones
FROM aplicacion_n a
JOIN lote l ON a.id_lote = l.id_lote
JOIN producto p ON a.id_producto = p.id_producto
JOIN operario_n op ON a.documento_operario = op.documento;

In [ ]:
%%sql
-- 1. Tendencia: plagas válidas por lote y semana (YEARWEEK)
SELECT
    i.id_lote,
    YEARWEEK(d.fecha_registro) AS semana,
    COUNT(d.id_deteccion) AS total_plagas
FROM deteccion d
JOIN imagen i ON d.id_imagen = i.id_imagen
WHERE d.confianza >= 0.45
GROUP BY i.id_lote, semana
ORDER BY i.id_lote, semana;

-- 2. Densidad: detecciones válidas de plaga por hectárea
SELECT
    l.codigo,
    l.cultivo,
    COUNT(d.id_deteccion) AS conteo_plagas,
    l.area_ha,
    ROUND(COUNT(d.id_deteccion) / l.area_ha, 2) AS plagas_por_ha
FROM deteccion d
JOIN imagen i ON d.id_imagen = i.id_imagen
JOIN lote l ON i.id_lote = l.id_lote
WHERE d.confianza >= 0.45
GROUP BY l.id_lote, l.codigo, l.cultivo, l.area_ha;

-- 3. Calidad del modelo: conteo de maleza por umbral de confianza
SELECT
    SUM(CASE WHEN confianza >= 0.30 THEN 1 ELSE 0 END) AS umbral_30,
    SUM(CASE WHEN confianza >= 0.45 THEN 1 ELSE 0 END) AS umbral_45,
    SUM(CASE WHEN confianza >= 0.60 THEN 1 ELSE 0 END) AS umbral_60,
    SUM(CASE WHEN confianza >= 0.75 THEN 1 ELSE 0 END) AS umbral_75
FROM deteccion
WHERE clase = 'maleza';

-- 4. Soporte multimodelo (v2)
ALTER TABLE deteccion
ADD COLUMN modelo_version VARCHAR(50) NOT NULL DEFAULT 'yolo8n-agro-v1';